# Loan Approval Prediction - Data Preprocessing

## Objective

The objective of this notebook is to preprocess the loan dataset before training machine learning models. This includes:

- Handling missing values
- Encoding categorical features
- Splitting the dataset
- Scaling the features
- Handling class imbalance using SMOTE

In [1]:
pip install imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import Required Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings("ignore")


In [3]:
df = pd.read_csv("../Dataset/loan_prediction.csv")

df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [4]:
df.drop("Loan_ID", axis=1, inplace=True)

df.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [5]:
df.isnull().sum()

Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

In [6]:
categorical_columns = [
    "Gender",
    "Married",
    "Dependents",
    "Self_Employed",
    "Credit_History"
]

for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [7]:
numerical_columns = [
    "LoanAmount",
    "Loan_Amount_Term"
]

for col in numerical_columns:
    df[col] = df[col].fillna(df[col].median())

In [8]:
df.isnull().sum()

Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

In [9]:
encoder = LabelEncoder()

categorical_cols = df.select_dtypes(include="object").columns

categorical_cols

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'Property_Area', 'Loan_Status'],
      dtype='str')

In [10]:
for col in categorical_cols:
    df[col] = encoder.fit_transform(df[col])

df.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,1,0,0,0,0,5849,0.0,128.0,360.0,1.0,2,1
1,1,1,1,0,0,4583,1508.0,128.0,360.0,1.0,0,0
2,1,1,0,0,1,3000,0.0,66.0,360.0,1.0,2,1
3,1,1,0,1,0,2583,2358.0,120.0,360.0,1.0,2,1
4,1,0,0,0,0,6000,0.0,141.0,360.0,1.0,2,1


In [11]:
X = df.drop("Loan_Status", axis=1)

y = df["Loan_Status"]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [13]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [14]:
smote = SMOTE(random_state=42)

X_train, y_train = smote.fit_resample(X_train, y_train)

In [15]:
print(y_train.value_counts())

Loan_Status
0    342
1    342
Name: count, dtype: int64


In [16]:
import pickle

with open("../models/scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

In [17]:
# Save processed data

X_train_df = pd.DataFrame(X_train)
X_test_df = pd.DataFrame(X_test)

X_train_df.to_csv("../Dataset/X_train.csv", index=False)
X_test_df.to_csv("../Dataset/X_test.csv", index=False)

pd.DataFrame(y_train).to_csv("../Dataset/y_train.csv", index=False)
pd.DataFrame(y_test).to_csv("../Dataset/y_test.csv", index=False)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


In [18]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = [
    "Gender",
    "Married",
    "Dependents",
    "Education",
    "Self_Employed",
    "Property_Area",
    "Loan_Status"
]

for col in categorical_cols:
    le = LabelEncoder()
    le.fit(df[col])
    print(f"\n{col}")
    print(dict(zip(le.classes_, le.transform(le.classes_))))


Gender
{np.int64(0): np.int64(0), np.int64(1): np.int64(1)}

Married
{np.int64(0): np.int64(0), np.int64(1): np.int64(1)}

Dependents
{np.int64(0): np.int64(0), np.int64(1): np.int64(1), np.int64(2): np.int64(2), np.int64(3): np.int64(3)}

Education
{np.int64(0): np.int64(0), np.int64(1): np.int64(1)}

Self_Employed
{np.int64(0): np.int64(0), np.int64(1): np.int64(1)}

Property_Area
{np.int64(0): np.int64(0), np.int64(1): np.int64(1), np.int64(2): np.int64(2)}

Loan_Status
{np.int64(0): np.int64(0), np.int64(1): np.int64(1)}


In [19]:
print(X.columns)

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area'],
      dtype='str')
